In [1]:
"""RAG PIPELINES - DATA INGESTION TO VECTOR DB PIPILINE"""

'RAG PIPELINES - DATA INGESTION TO VECTOR DB PIPILINE'

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/tmp/ipykernel_26963/3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
/home/boa/miniconda3/envs/rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
###Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""

    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents  

# Process all PDFs in the data direcotry
all_pdf_documents = process_all_pdfs("data")

Found 3 PDF files to process

Processing: NIST_GenAI_Profile.pdf
 Loaded 64 pages

Processing: 1103-Policy-Statement-Rule-of-Law-Principles.pdf
 Loaded 6 pages

Processing: NIST_AI_RMF.pdf
 Loaded 48 pages

Total documents loaded: 118


In [4]:
###Text splitting get into chunks


def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Splitting documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
        
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    ## Showing an example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 118 documents into 384 chunks

Example chunk:
Content: NIST Trustworthy and Responsible AI  
NIST AI 600-1 
Artificial Intelligence Risk Management 
Framework: Generative Artificial 
Intelligence Profile 
 
 
 
This publication is available free of charge
Metadata: {'producer': 'Adobe PDF Library 24.2.159', 'creator': 'Acrobat PDFMaker 24 for Word', 'creationdate': '2024-08-05T14:17:02-04:00', 'author': 'National Institute of Standards and Technology', 'category': 'NIST AI 600-1', 'comments': '', 'company': '', 'contenttypeid': '0x01010068CEA9BE6E0AF749888425167690E526', 'keywords': '', 'mediaserviceimagetags': '', 'moddate': '2025-03-24T15:11:27-04:00', 'sourcemodified': 'D:20240805160221', 'subject': '', 'title': 'Artificial Intelligence Risk Management Framework: Generative Artificial Intelligence Profile', 'source': 'data/pdf/NIST_GenAI_Profile.pdf', 'total_pages': 64, 'page': 0, 'page_label': '1', 'source_file': 'NIST_GenAI_Profile.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Adobe PDF Library 24.2.159', 'creator': 'Acrobat PDFMaker 24 for Word', 'creationdate': '2024-08-05T14:17:02-04:00', 'author': 'National Institute of Standards and Technology', 'category': 'NIST AI 600-1', 'comments': '', 'company': '', 'contenttypeid': '0x01010068CEA9BE6E0AF749888425167690E526', 'keywords': '', 'mediaserviceimagetags': '', 'moddate': '2025-03-24T15:11:27-04:00', 'sourcemodified': 'D:20240805160221', 'subject': '', 'title': 'Artificial Intelligence Risk Management Framework: Generative Artificial Intelligence Profile', 'source': 'data/pdf/NIST_GenAI_Profile.pdf', 'total_pages': 64, 'page': 0, 'page_label': '1', 'source_file': 'NIST_GenAI_Profile.pdf', 'file_type': 'pdf'}, page_content='NIST Trustworthy and Responsible AI  \nNIST AI 600-1 \nArtificial Intelligence Risk Management \nFramework: Generative Artificial \nIntelligence Profile \n \n \n \nThis publication is available free of charge from: \nhttps://doi.org/10.6028/NIST.AI.600-1'

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity



In [7]:
class EmbeddingManager:

    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """

        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings

        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:  

        """
        Generate embeddings for a list of texts

        Args: 
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
            """

        if not self.model:
            raise ValueError("Model not loaded")
    
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings 

## initilialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1883.67it/s]


Model loaded successfully. Embedding dimension: 384


/tmp/ipykernel_26963/2642550724.py:23: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore


In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = 'pdf_documents', persist_directory: str = "data/vector_store"):

        """ 
        Intilaize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
            
        """
    
        self.collection_name = collection_name
        self.persist_directory = persist_directory        
        self.client = None
        self.collection = None
        self._initialize_store()


    def _initialize_store(self):
            """Initialize ChromaDB client and collection"""
            try:
                #Create persistent ChromaDB client
                os.makedirs(self.persist_directory, exist_ok=True)
                self.client = chromadb.PersistentClient(path=self.persist_directory)

                #Get or create collection
                self.collection = self.client.get_or_create_collection(
                    name=self.collection_name,
                    metadata={"description": "PDF document embeddings for RAG"}


                )
                print(f"Vector store initialized. Collection: {self.collection_name}")
                print(f"Existing documents in collection: {self.collection.count()}")

            except Exception as e:
                print(f"Error initializing vector store: {e}")
                raise 

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
            """Add documents and their embeddings to the vector store

                args:
                    documents: List of LangChain documents
                    embeddings: Corresponding embeddings for the documents
            """
            if len(documents) != len(embeddings):
                raise ValueError("Number of documents must match the number of embeddings")
            
            print(f"Adding {len(documents)} documents to vector store...")

            # Prepre data for ChromaDB
            ids = []
            metadatas = []
            documents_text = []
            embeddings_list = []

            for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
                # Generate unique ID
                doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
                ids.append(doc_id)

                #Prepare metadata
                metadata = dict(doc.metadata)
                metadata['doc_index'] = i
                metadata['content_length'] = len(doc.page_content)
                metadatas.append(metadata)

                # Document content
                documents_text.append(doc.page_content)

                # Embedding
                embeddings_list.append(embedding.tolist())

            #Add to collection
            try:
                self.collection.add(
                    ids=ids,
                    embeddings=embeddings_list,
                    metadatas=metadatas,
                    documents=documents_text
                )
                print(f"Successfuly added {len(documents)} documents to vector store")
                print(f"Total documents in collection: {self.collection.count()}")

            except Exception as e:
                print(f"Error adding documents to vector store: {e}")
                raise


vectorstore=VectorStore()
vectorstore
        

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [9]:
chunks

[Document(metadata={'producer': 'Adobe PDF Library 24.2.159', 'creator': 'Acrobat PDFMaker 24 for Word', 'creationdate': '2024-08-05T14:17:02-04:00', 'author': 'National Institute of Standards and Technology', 'category': 'NIST AI 600-1', 'comments': '', 'company': '', 'contenttypeid': '0x01010068CEA9BE6E0AF749888425167690E526', 'keywords': '', 'mediaserviceimagetags': '', 'moddate': '2025-03-24T15:11:27-04:00', 'sourcemodified': 'D:20240805160221', 'subject': '', 'title': 'Artificial Intelligence Risk Management Framework: Generative Artificial Intelligence Profile', 'source': 'data/pdf/NIST_GenAI_Profile.pdf', 'total_pages': 64, 'page': 0, 'page_label': '1', 'source_file': 'NIST_GenAI_Profile.pdf', 'file_type': 'pdf'}, page_content='NIST Trustworthy and Responsible AI  \nNIST AI 600-1 \nArtificial Intelligence Risk Management \nFramework: Generative Artificial \nIntelligence Profile \n \n \n \nThis publication is available free of charge from: \nhttps://doi.org/10.6028/NIST.AI.600-1'

In [10]:
### Convert the text to embeddings...
texts=[doc.page_content for doc in chunks]


# Generate the embeddings

embeddings=embedding_manager.generate_embeddings(texts)

#tore in the vector db
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 384 texts...


Batches: 100%|██████████| 12/12 [00:17<00:00,  1.43s/it]


Generated embeddings with shape: (384, 384)
Adding 384 documents to vector store...
Successfuly added 384 documents to vector store
Total documents in collection: 384


"""Retrieval pipeline form vector db


In [17]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Intialize the retriever class

        Args:
        vector_store: vector store containing document embeddings
        embedding_manager: manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:

        """
        Retrieve relevant documents for a query

        Args: 
            query: the search query
            top_k: no of top results to return
            score_threshold: minimum similarity score threshold

        returns:
            List of dictionaries containing retrieved documents and metadata
            """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k

            )

            # process the results gotten from the search
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # converting distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever= RAGRetriever(vectorstore,embedding_manager)


In [18]:
rag_retriever.retrieve("What are the risks of generative AI?")

Retrieving documents for query: 'What are the risks of generative AI?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 32.40it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_bf9cc56b_256',
  'content': 'some AI risks and benefits are well-known, it can be challenging to assess negative impacts\nand the degree of harms. Figure 1 provides examples of potential harms that can be related\nto AI systems.\nAI risk management efforts should consider that humans may assume that AI systems work\n– and work well – in all settings. For example, whether correct or not, AI systems are\noften perceived as being more objective than humans or as offering greater capabilities\nthan general software.\nPage 4',
  'metadata': {'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4',
   'content_length': 489,
   'doc_index': 256,
   'moddate': '2025-06-04T13:01:45-04:00',
   'producer': 'pdfTeX-1.40.24',
   'keywords': 'Artificial Intelligence (AI); AI; AI RMF; AI RMF 1.0; AI systems; trustworthy and responsible AI.',
   'page_label': '4',
   'file_type': 'pdf',
   'subject': 'As directed by the National Artifici

Integration VectorDB context pipeline with LLM output

In [13]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)

## 2. Simple  RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retrieve the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return " NO relevant context found to answer the question. "
    
    ## generating the answer using GROQ LLM
    prompt=f"""Use the folllowing context to answer the question concisely.
    Context:
    {context}

    Question: {query}

    Answer:"""

    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content


In [14]:
answer=rag_simple("What are the risks of generative AI?",rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What are the risks of generative AI?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 49.84it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


The risks of Generative AI (GAI) can vary along many dimensions, including:

1. Stage of the AI lifecycle (design, development, deployment, operation, and decommissioning)
2. Scope (individual model, system, application, implementation, or ecosystem level)
3. Source of risk (design, training, or operation of the GAI model, or from inputs)

These risks can include:
- Exacerbating existing AI risks
- Creating unique risks
- Intensifying traditional software risks
- Algorithmic monocultures
- Impacts on access to opportunity, labor markets, and creative economies
- Biases (even worse than human biases)


Advanced RAG pipeline features

In [15]:
def rag_advanced(query,retriever,llm,top_k=3, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
        - Returns answer, sources, confidence score, and optionally full context.
    """

    ## retrieve the context
    results=retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    #preparing the context and sources
    context="\n\n".join([doc['content'] for doc in results]) 
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    #generate answer
    prompt = f"Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    response = llm.invoke(prompt)

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context'] = context
    return output
    
#example usage
result = rag_advanced('What are the risks of generative AI?', rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What are the risks of generative AI?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 75.75it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: The risks of Generative AI (GAI) can vary along many dimensions, including:

1. Stage of the AI lifecycle (design, development, deployment, operation, and decommissioning)
2. Scope (individual model, system, application, implementation, or ecosystem level)
3. Source of risk (design, training, or operation of the GAI model, or from model inputs)

These risks can include:
- Exacerbating existing AI risks
- Creating unique risks
- Intensifying traditional software risks
- Algorithmic monocultures
- Impacts on access to opportunity, labor markets, and creative economies
- Biases (even worse than human biases) 

Note: These risks may be uncertain and speculative, and can differ from traditional software risks.
Sources: [{'source': 'NIST_AI_RMF.pdf', 'page': 8, 'score': 0.22887438535690308, 'preview': 'some AI risks and benefits are well-known, it can be challenging to assess negative impacts\nand the degree of harms. Figure 1 provides examples of potential harms that can be related\

ENHANCED RAG retriever


In [16]:
# --- Enhanced RAG pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class EnhancedRAGpipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = [] #Stores query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False):
        #Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found"
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely. \nContext: \n{context}\n\nQuestion: {question}\n\nAnswer: """
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke(prompt)
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = EnhancedRAGpipeline(rag_retriever, llm)
result = adv_rag.query("What are the risks of generative AI?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])


                

        

Retrieving documents for query: 'What are the risks of generative AI?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 61.23it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely. 
Context: 
some AI risks and benefits are well-known, it can be challenging to assess negative impacts
and the degree of harms. Figure 1 provides examples of potential harms that can be related
to AI systems.
AI risk management efforts should consider that hum

ans may assume that AI systems work
– and work well – in all settings. For example, whether correct or not, AI systems are
often perceived as being more objective than humans or as offering greater capabilities
than general software.
Page 4

https://www.rand.org/pubs/research_reports/RRA2977-2.html. 
Nicoletti, L. et al. (2023) Humans Are Biased. Generative Ai Is Even Worse. Bloomberg. 
https://www.bloomberg.com/graphics/2023-generative-ai-bias/. 
National Institute of Standards and Technology (2024) Adversarial Machine Learning: A Taxonomy and 
Terminology of Attacks and Mitigations https://csrc.nist.gov/pubs/ai/100/2/e2023/ﬁnal 
National Institute of Standards and Technology (2023) AI Risk Management Framework. 
https://www.nist.gov/itl/ai-risk-management-framework 
National Institute of Standards and Technology (2023) AI Risk Management Framework, Chapter 3: AI 
Risks and Trustworthiness. 
https://airc.nist.gov/AI_RMF_Knowledge_Base/AI_RMF/Foundational_Information/3-sec-characterist

"" embedding and vectorstoreDB